# Weather Engine

## Purpose

The Weather Engine retrieves atmospheric conditions for the selected Area of Interest (AOI) and generates standardized weather information for Earth Intelligence analyses.

Version 1 retrieves:

- Air Temperature
- Precipitation
- Wind Speed
- Wind Direction
- Relative Humidity
- Surface Pressure

The resulting Weather Product provides environmental context for downstream modules including agriculture, disaster management, environmental monitoring, and risk assessment.

# Import Libraries

## Purpose

Import the libraries required for weather data retrieval, processing, and analysis.

In [1]:
from pathlib import Path
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

import requests

import matplotlib.pyplot as plt

# Load Inputs

## Purpose

Load the input datasets required by the Weather Engine.

The Weather Engine uses:

- Area of Interest (AOI)
- Earth Intelligence Catalog

These datasets provide the spatial boundary and weather dataset information needed for weather retrieval.

In [2]:
from pathlib import Path
import json
import geopandas as gpd

DATA_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

# Load Area of Interest
aoi_file = DATA_DIR / "aoi.geojson"

if not aoi_file.exists():
    raise FileNotFoundError(
        "AOI not found. Run Notebook 01 first."
    )

aoi_boundary = gpd.read_file(aoi_file)

# Load Earth Intelligence Catalog
catalog_file = DATA_DIR / "catalog.json"

if not catalog_file.exists():
    raise FileNotFoundError(
        "Catalog not found. Run Notebook 02 first."
    )

with open(catalog_file, "r", encoding="utf-8") as file:
    catalog = json.load(file)

print("Inputs loaded successfully.")

Inputs loaded successfully.


# Weather Configuration

## Purpose

Define the weather retrieval settings used by the Weather Engine.

The configuration specifies:

- Weather provider
- Temporal mode
- Date range
- Temporal resolution
- Weather variables

Centralizing these settings allows the Weather Engine to be configured without modifying the retrieval logic.

In [3]:
from datetime import date, timedelta

today = date.today()

weather_configuration = {

    "provider": "Open-Meteo",

    "mode": "historical",

    "start_date": (
        today - timedelta(days=30)
    ).isoformat(),

    "end_date": (
        today - timedelta(days=1)
    ).isoformat(),

    "temporal_resolution": "daily",

    "variables": [

        "temperature_2m_mean",

        "precipitation_sum",

        "relative_humidity_2m_mean",

        "surface_pressure_mean",

        "wind_speed_10m_max",

        "wind_direction_10m_dominant"

    ]

}

# Weather Dataset Selection

## Purpose

Select the preferred weather dataset from the Earth Intelligence Catalog.

The selection is based on dataset metadata rather than hardcoded provider names.

Selection Criteria

- Category = Weather
- Applicable = True
- Highest Priority

The selected dataset becomes the source for weather retrieval.

In [4]:
weather_datasets = [

    dataset

    for dataset in catalog["datasets"]

    if dataset["category"] == "Weather"
    and dataset["applicable"]

]

if len(weather_datasets) == 0:

    raise ValueError(
        "No applicable weather datasets found."
    )

priority_order = {
    "Primary": 1,
    "Secondary": 2
}

weather_datasets = sorted(

    weather_datasets,

    key=lambda dataset: priority_order.get(
        dataset["priority"],
        99
    )

)

selected_dataset = weather_datasets[0]

selected_dataset

{'id': 'open_meteo',
 'name': 'Open-Meteo Weather API',
 'category': 'Weather',
 'provider': 'Open-Meteo',
 'access_method': 'API',
 'description': 'Global weather API providing historical, current, and forecast weather data.',
 'coverage': 'Global',
 'spatial_resolution': 'Point',
 'temporal_resolution': 'Hourly / Daily',
 'data_type': 'Time Series',
 'applicable': True,
 'priority': 'Primary',
 'notes': 'Preferred weather dataset for Version 1.'}

# Retrieve Weather Data

## Purpose

Retrieve weather observations for the selected Area of Interest (AOI).

The Weather Engine uses the centroid of the AOI as the representative location for weather retrieval.

Version 1 retrieves daily weather observations from the Open-Meteo Archive API.

In [5]:
# AOI centroid
centroid = aoi_boundary.geometry.iloc[0].centroid

latitude = centroid.y
longitude = centroid.x

weather_url = "https://archive-api.open-meteo.com/v1/archive"

parameters = {

    "latitude": latitude,

    "longitude": longitude,

    "start_date": weather_configuration["start_date"],

    "end_date": weather_configuration["end_date"],

    "daily": ",".join(
        weather_configuration["variables"]
    ),

    "timezone": "auto"

}

response = requests.get(
    weather_url,
    params=parameters
)

response = requests.get(
    weather_url,
    params=parameters
)

if not response.ok:
    print(response.text)
    response.raise_for_status()

weather_data = response.json()

weather_data = response.json()

weather_data.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

# Generate Weather Layers

## Purpose

Convert the retrieved weather observations into a structured weather dataset.

The Weather Layers dataset provides a tabular representation of daily weather observations for the selected Area of Interest.

It serves as the primary data structure for weather analysis and for populating the Weather Product.

In [6]:
weather_layers = pd.DataFrame(
    weather_data["daily"]
)

weather_layers["time"] = pd.to_datetime(
    weather_layers["time"]
)

weather_layers

,time,temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_max,wind_direction_10m_dominant
0,2026-06-15,31.0,2.5,65,1007.4,20.4,271
1,2026-06-16,31.3,0.6,64,1007.4,18.6,257
2,2026-06-17,31.3,0.8,61,1007.9,21.8,258
3,2026-06-18,31.1,1.5,62,1007.5,16.8,251
4,2026-06-19,31.2,0.3,60,1007.3,18.2,232
5,2026-06-20,31.3,0.0,61,1007.6,15.9,246
6,2026-06-21,31.0,0.5,63,1006.4,15.2,255
7,2026-06-22,29.5,14.9,72,1005.8,16.3,200
8,2026-06-23,26.6,74.4,89,1004.8,15.4,189
9,2026-06-24,26.8,32.5,85,1005.3,15.2,179


# Weather Summary

## Purpose

Summarize the retrieved weather observations.

The summary provides a quick overview of the weather conditions within the selected time period and validates the retrieved dataset before creating the Weather Product.

In [7]:
weather_summary = {

    "Start Date": weather_layers["time"].min().date(),

    "End Date": weather_layers["time"].max().date(),

    "Average Temperature (°C)": round(
        weather_layers["temperature_2m_mean"].mean(),
        2
    ),

    "Total Precipitation (mm)": round(
        weather_layers["precipitation_sum"].sum(),
        2
    ),

    "Average Relative Humidity (%)": round(
        weather_layers["relative_humidity_2m_mean"].mean(),
        2
    ),

    "Average Surface Pressure (hPa)": round(
        weather_layers["surface_pressure_mean"].mean(),
        2
    ),

    "Average Wind Speed (km/h)": round(
        weather_layers["wind_speed_10m_max"].mean(),
        2
    )

}

weather_summary

{'Start Date': datetime.date(2026, 6, 15),
 'End Date': datetime.date(2026, 7, 14),
 'Average Temperature (°C)': np.float64(28.57),
 'Total Precipitation (mm)': np.float64(906.4),
 'Average Relative Humidity (%)': np.float64(78.9),
 'Average Surface Pressure (hPa)': np.float64(1005.02),
 'Average Wind Speed (km/h)': np.float64(21.52)}

# Weather Product

## Purpose

Create and populate the standardized Weather Product.

The Weather Product summarizes the weather dataset, retrieved weather observations, and processing metadata generated by the Weather Engine.

It serves as the primary output of the Weather Engine and provides weather information for downstream Earth Intelligence modules.

In [8]:
from datetime import datetime

weather_product = {

    "metadata": {

        "engine": "Weather Engine",

        "version": "1.0",

        "created_at": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    },

    "dataset": {

        "id": selected_dataset["id"],

        "name": selected_dataset["name"],

        "provider": selected_dataset["provider"],

        "category": selected_dataset["category"],

        "access_method": selected_dataset["access_method"]

    },

    "weather": {

        "start_date": str(weather_layers["time"].min().date()),

        "end_date": str(weather_layers["time"].max().date()),

        "variables": weather_configuration["variables"],

        "records": len(weather_layers)

    },

    "location": {

        "latitude": latitude,

        "longitude": longitude

    },

    "processing": {

        "provider": weather_configuration["provider"],

        "mode": weather_configuration["mode"],

        "temporal_resolution": weather_configuration["temporal_resolution"]

    }

}

weather_product

{'metadata': {'engine': 'Weather Engine',
  'version': '1.0',
  'created_at': '2026-07-15 20:28:43'},
 'dataset': {'id': 'open_meteo',
  'name': 'Open-Meteo Weather API',
  'provider': 'Open-Meteo',
  'category': 'Weather',
  'access_method': 'API'},
 'weather': {'start_date': '2026-06-15',
  'end_date': '2026-07-14',
  'variables': ['temperature_2m_mean',
   'precipitation_sum',
   'relative_humidity_2m_mean',
   'surface_pressure_mean',
   'wind_speed_10m_max',
   'wind_direction_10m_dominant'],
  'records': 30},
 'location': {'latitude': 18.980765177226417, 'longitude': 72.83380382195719},
 'processing': {'provider': 'Open-Meteo',
  'mode': 'historical',
  'temporal_resolution': 'daily'}}

# Export Weather Product

## Purpose

Export the outputs generated by the Weather Engine.

The Weather Product metadata is exported as a JSON file.

The retrieved weather observations are exported as a CSV file for downstream Earth Intelligence modules.

In [11]:
from pathlib import Path
import json

OUTPUT_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Export Weather Product
weather_product_file = OUTPUT_DIR / "weather_product.json"

with open(weather_product_file, "w", encoding="utf-8") as file:
    json.dump(
        weather_product,
        file,
        indent=4,
        default=str
    )

# Export Weather Layers
weather_layers_file = OUTPUT_DIR / "weather_layers.csv"

weather_layers.to_csv(
    weather_layers_file,
    index=False
)

print(f"✓ Weather Product : {weather_product_file.name}")
print(f"✓ Weather Layers  : {weather_layers_file.name}")

✓ Weather Product : weather_product.json
✓ Weather Layers  : weather_layers.csv


# Weather Engine Summary

## Purpose

Summarize the outputs generated by the Weather Engine.

This summary confirms that the Weather Engine successfully retrieved and standardized the weather observations for the selected Area of Interest.

In [12]:
summary = {

    "Dataset": weather_product["dataset"]["name"],

    "Period": (
        f"{weather_product['weather']['start_date']} to "
        f"{weather_product['weather']['end_date']}"
    ),

    "Records": weather_product["weather"]["records"],

    "Variables": len(weather_product["weather"]["variables"]),

    "Provider": weather_product["processing"]["provider"]

}

for key, value in summary.items():
    print(f"{key:<12}: {value}")

Dataset     : Open-Meteo Weather API
Period      : 2026-06-15 to 2026-07-14
Records     : 30
Variables   : 6
Provider    : Open-Meteo
